# Setup & Installation

In [1]:
! pip install pandas sentence-transformers faiss-cpu
! pip install rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.6 MB/s eta 0:00:00:00:0100:01


# Import

In [2]:
import os
import pandas as pd
import numpy as np
import faiss
import pickle

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm import tqdm
from sentence_transformers import CrossEncoder

2026-02-07 10:49:56.065796: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770461396.442772      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770461396.552318      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770461397.459290      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770461397.459326      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770461397.459329      55 computation_placer.cc:177] computation placer alr

# Load the data

In [3]:
movies = pd.read_csv('/kaggle/input/all-data-movies/final_merged_movies.csv')
print(movies.shape)

(4801, 23)


In [4]:
movies.columns

Index(['budget', 'homepage', 'id', 'original_language', 'original_title',
       'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'vote_average', 'vote_count',
       'Poster', 'on_netflix', 'on_disney', 'on_amazon', 'on_hulu',
       'search_text'],
      dtype='object')

# Embeddings & vectior dataset

## Embedding

In [5]:
print("Initializing Embedding Model")
embedder = SentenceTransformer('all-mpnet-base-v2')

Initializing Embedding Model


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
print("Creating Vector Index - Convert movie descriptions to vectors")
movie_vectors = embedder.encode(movies['search_text'].tolist(), show_progress_bar=True)

Creating Vector Index - Convert movie descriptions to vectors


Batches:   0%|          | 0/151 [00:00<?, ?it/s]

## Save movie vectors

In [7]:
np.save('movie_vectors.npy', movie_vectors)

 ## Build FAISS

In [8]:
print("Build FAISS Index for fast retrieval")
dimension = movie_vectors.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(movie_vectors))
faiss.write_index(index, "movie_faiss.index")

Build FAISS Index for fast retrieval


## Build BM25

In [9]:
tokenized_corpus = [doc.split(" ") for doc in movies['search_text']]
bm25 = BM25Okapi(tokenized_corpus)
with open('bm25_model.pkl', 'wb') as f:
    pickle.dump(bm25, f)